# AI Trainer: QLoRA fine-tune of the coach brain (Kaggle)
**Before running:** right sidebar -> *Session options* -> **Accelerator: GPU T4 x2 (or P100)** and **Internet: On** (needs a phone-verified Kaggle account).
Just *Run All*. It clones `github.com/jhmle/fitness` automatically. Takes roughly 15-30 minutes.

In [ ]:
REPO_URL = "https://github.com/jhmle/fitness.git"
!rm -rf /kaggle/working/repo && git clone --depth 1 {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo/training
!pip install -q -U peft bitsandbytes accelerate transformers mlflow scikit-learn
!pip uninstall -y -q torchao   # Kaggle's preinstalled torchao 0.10 is too old for the new peft and breaks adapter loading

# Optional: authenticate to Hugging Face (faster downloads, no rate-limit warning). Skipped if the HF_TOKEN secret isn't set.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret found: continuing unauthenticated")

## 1. Data
Parses the prompt + knowledge base straight from `web/agent.js`, simulates reps with NumPy. Add `--logs` with your downloaded session logs to mix in real states.

In [ ]:
!python gen_data.py

## 2. Baseline: the untuned base model (so the fine-tune has a number to beat)

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --name base

## 3. QLoRA fine-tune (4-bit NF4 + LoRA r=32, loss on the JSON answer only)

In [ ]:
!python finetune_qlora.py --epochs 4 --lora-r 32 --lr 1e-4

## 4. Evaluate the fine-tuned model on the frozen test set (+ log adapter to MLflow and register a version)

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --adapter out/adapter --name candidate --register trainer-coach

## 5. Compare against production (same gate CI runs)

In [ ]:
!python gate.py --candidate metrics/candidate.json --production metrics/production.json --min-valid 0.95

## 6. (Optional) merge + push to the Hugging Face Hub for ONNX conversion
Needs a Kaggle secret named `HF_TOKEN` (a Hugging Face **write** token for the `jhmle` account; Add-ons -> Secrets). Pushes to `jhmle/trainer-coach-smollm2-360m`. Skip this step if you haven't set it up.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # must be a WRITE token
HF_REPO = "jhmle/trainer-coach-smollm2-360m"
!python merge_export.py --adapter out/adapter --out out/merged --push {HF_REPO}

## 7. Package results
Download `results.zip` from the *Output* tab, then commit `metrics/candidate.json` to your repo.

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip training/metrics training/out/adapter mlflow.db && ls -lh /kaggle/working/results.zip